## Security groups vs. network ACLs

A **security group** is a virtual firewall attached to an **ENI** (an instance, load balancer, RDS, VPC-Lambda — anything with a VPC IP). Four properties define it: it's **stateful** (allow inbound 443 and the return is allowed automatically — no matching outbound rule needed); it's **allow-only** (no deny; everything not permitted is implicitly denied); **multiple SGs per ENI** combine **additively** (if any allows, it's allowed); and rules can **reference another SG** as the source — "allow 8080 from the ALB's SG" tracks instances as they scale, far better than an IP range.

A **NACL** sits one layer up, at the **subnet** boundary, and has **inverted** properties: it's **stateless** (return traffic is *not* auto-allowed), supports **allow *and* deny**, and evaluates rules in **order, first match wins**. The default NACL allows all; a custom NACL denies all until you add rules.

The stateless property is the classic trap: because responses aren't auto-allowed, you must permit **both directions** — and a response comes back on an **ephemeral port** (Linux 1024–65535). Allow inbound 443 but forget outbound on the ephemeral range and the handshake completes, then hangs. **Rule of thumb: security groups do 99% of the work; reach for a NACL only to *deny* something (an IP block) at the subnet edge.**